In [ ]:
from platform import python_version
print(python_version())

### Open Access

- disease 
- drug_mechanism_of_action 
- evidence_cancer_gene_census 
- evidence_reactome 
- interactions 
- pharmacogenomics 
- study

https://www.opentargets.org/

find a gene:
https://platform.opentargets.org/target/ENSG00000153395/associations

download:
https://platform.opentargets.org/downloads

In [ ]:
import os, sys, yaml
from pathlib import Path
from dotenv import load_dotenv

import numpy as np
import pandas as pd
pd.set_option('display.width', 100)
pd.set_option('max_colwidth', 80)
pd.set_option("display.precision", 3)

import seaborn as sns
sns.set_context("notebook", font_scale=1.4)

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline

sys.path.insert(1, '../src/')

ROOT0 = Path("/home/flavio/uv/perturb_agent/")
ROOT_SRC = ROOT0 / "src"


if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

print("ROOT0:", ROOT0)
print("ROOT_SRC added:", ROOT_SRC)


from libs.Basic import pdwritecsv, pdreadcsv, create_dir, write_txt
from libs.enricher_lib import enricheR
from libs.config_lib import Config
s
from IPython.display import display, HTML
# display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("<style>:root_ot { --jp-notebook-max-width: 100% !important; }</style>"))

with open('../params.yml', 'r') as file:
    dic_yml = yaml.safe_load(file)

# print(dic_yml)

In [ ]:
email = os.getenv('email')

i_project=0

project_list = dic_yml['project_list']
n = len(project_list)
project = project_list[i_project]

s_project_list = dic_yml['s_project_list']
s_project = s_project_list[i_project]
assert n==len(project_list), f"Error project_list: there are {n} projects"

PROG_ID = 'TCGA'
PSI_ID = 'TCGA-BRCA'
PSI_ID = 'TCGA-ACC'
PSI_ID = 'TCGA-CESC'

disease = PSI_ID

ROOT0_DATA = ROOT0 / "data"
root_colab = ROOT0_DATA / 'colab'
root_project = ROOT0_DATA / PROG_ID
root_disease = create_dir(root_project, PSI_ID)

CONTEXT_DISESE = 'xxxx'
context_disease = CONTEXT_DISESE

gene_protein = dic_yml['gene_protein']
s_omics = dic_yml['s_omics']

has_age = dic_yml['has_age']
has_gender = dic_yml['has_gender']

exp_normalization = dic_yml['exp_normalization']
normalization = 'quantile_norm' if exp_normalization == True else 'not_normalized'

LFC_cut_inf = dic_yml['LFC_cut_inf']
s_pathw_enrichm_method = dic_yml['s_pathw_enrichm_method']
ptw_min_num_of_degs_cut = dic_yml['ptw_min_num_of_degs_cut']

tolerance_pPMI = dic_yml['tolerance_pPMI']
type_sat_ptw_index = dic_yml['type_sat_ptw_index']
saturation_lfc_param = dic_yml['saturation_lfc_param']

pval_pathway_cutoff = dic_yml['pval_pathway_cutoff']
fdr_pathway_cutoff = dic_yml['fdr_pathway_cutoff']
num_of_genes_cutoff = dic_yml['num_of_genes_cutoff']
enr_db_list = dic_yml['enr_db_list']


case_list = dic_yml['case_list']
dic_case_list = dic_yml['dic_case_list']

std_filename      = dic_yml['std_filename']
std_filename_list = dic_yml['std_filename_list']

min_lfc_modulation = dic_yml['min_lfc_modulation']
num_of_genes_list  = dic_yml['num_of_genes_list']
pPMI_normalized  = dic_yml['pPMI_normalized']

#--- max len for formatting purposes
s_len_case  = dic_yml['s_len_case']

n_sentences = dic_yml['n_sentences']
run_list = dic_yml['run_list']
chosen_model_list = dic_yml['chosen_model_list']
i_dfp_list = dic_yml['i_dfp_list']
chosen_model_sampling = dic_yml['chosen_model_sampling']

fdr_ptw_cutoff_list = np.arange(0.05, 0.80, 0.05)
lfc_list = np.round(np.arange(1.0, -0.01, -.025), 3)
fdr_list = np.arange(0.05, 0.76, .01)

cfg = Config(root0=ROOT0, root_disease=root_disease, disease=disease, case_list=case_list)
case = case_list[0]

n_genes_annot_ptw, n_degs, n_degs_in_ptw, n_degs_not_in_ptw, degs_in_all_ratio = -1,-1,-1,-1,-1

LFC_cut, lfc_FDR_cut, n_degs, n_degs_up, n_degs_dw = cfg.get_best_lfc_cutoff(case, 'not_normalized')

print(f"project '{project}', s_project '{s_project}'")
print(f"G/P LFC cutoffs: lfc={LFC_cut:.3f}; fdr={lfc_FDR_cut:.3f} - LFC_cut_inf={LFC_cut_inf:.3f}")
print(f"Pathway cutoffs: pval={pval_pathway_cutoff:.3f}; fdr={fdr_pathway_cutoff:.3f}; num of genes={num_of_genes_cutoff}")

In [ ]:
enr = enricheR(disease=disease, gene_protein=gene_protein, s_omics=s_omics, project=project, s_project=s_project, 
               root0=ROOT0, root0_data=ROOT0_DATA, prog_id=PROG_ID, psi_id=PSI_ID,
               case_list=case_list, dic_case_list=dic_case_list, has_age=has_age, has_gender=has_gender, exp_normalization=exp_normalization,
               std_filename=std_filename, std_filename_list=std_filename_list,
               geneset_num=0, ptw_min_num_of_degs_cut=ptw_min_num_of_degs_cut,
               tolerance_pPMI=tolerance_pPMI, s_pathw_enrichm_method=s_pathw_enrichm_method,
               LFC_cut_inf=LFC_cut_inf, fdr_ptw_cutoff_list=fdr_ptw_cutoff_list,
               num_of_genes_list=num_of_genes_list, lfc_list=lfc_list, fdr_list=fdr_list, 
               min_lfc_modulation=min_lfc_modulation, type_sat_ptw_index=type_sat_ptw_index,
               saturation_lfc_param=saturation_lfc_param, enr_db_list=enr_db_list, pPMI_normalized=pPMI_normalized)

print(">>> Roots", enr.root0, enr.root_disease)
case = case_list[0]
print(">>>", case)

enr.cfg.set_default_best_lfc_cutoff(enr.normalization, LFC_cut=1, lfc_FDR_cut=0.05)
ret, degs, degs_ensembl, dfdegs = enr.open_case(case, prompt_verbose=True, verbose=False)
print("\nEcho Parameters:")
print(enr.echo_parameters())


In [ ]:
enr.root_kegg, enr.kegg_fname

In [ ]:
enr.enr_db_list

In [ ]:
print(len(enr.gene.df_my_gene))
enr.gene.df_my_gene.head(2)

### Keneddy Pathway

https://www.wikipathways.org/pathways/WP3933.html

![Kennedy Pathway](../figures/WP3933_kennety_pathway.svg)

### Open Targets

```Bash
rsync -rpltvz --delete rsync.ebi.ac.uk::pub/databases/opentargets/platform/26.03/output/evidence_cancer_gene_census .
rsync -rpltvz --delete rsync.ebi.ac.uk::pub/databases/opentargets/platform/26.03/output/drug_mechanism_of_action .
rsync -rpltvz --delete rsync.ebi.ac.uk::pub/databases/opentargets/platform/26.03/output/pharmacogenomics .
rsync -rpltvz --delete rsync.ebi.ac.uk::pub/databases/opentargets/platform/26.03/output/study .
rsync -rpltvz --delete rsync.ebi.ac.uk::pub/databases/opentargets/platform/26.03/output/evidence_reactome .
rsync -rpltvz --delete rsync.ebi.ac.uk::pub/databases/opentargets/platform/25.03/output/association_by_datasource_direct .
rsync -rpltvz --delete rsync.ebi.ac.uk::pub/databases/opentargets/platform/25.03/output/target .

colab/
    open_targets/
    ├── target/
    ├── disease/
    ├── drug_mechanism_of_action/
    ├── evidence_cancer_gene_census/
    ├── evidence_reactome/
    ├── interactions/
    ├── pharmacogenomics/
    └── study/
```

In [ ]:
from pathlib import Path
import duckdb
import pandas as pd

root_ot = root_colab / 'open_targets'

### Is target ok?

In [ ]:
root_target =  root_ot / "target"

print("exists:", root_target.exists())
print("is dir:", root_target.is_dir())

i=0
for p in root_target.rglob("*"):
    print(p)
    i+=1
    if i>4: break

In [ ]:
print("root exists:", root_ot.exists())
print("root is dir:", root_ot.is_dir())

print("\nTop-level content:")
for p in sorted(root_ot.iterdir()):
    print(
        "DIR " if p.is_dir() else "FILE",
        p.name
    )

In [ ]:
class OpenTargetsLocal:
    def __init__(self, root_ot: Path | Path):
        self.root_ot = root_ot
        self.con = duckdb.connect()

        self.tables = {
            "target": self.root_ot / "target",
            "disease": self.root_ot / "disease",
            "drug_moa": self.root_ot / "drug_mechanism_of_action",
            "cgc": self.root_ot / "evidence_cancer_gene_census",
            "reactome": self.root_ot / "evidence_reactome",
            "interactions": self.root_ot / "interactions",
            "pharmacogenomics": self.root_ot / "pharmacogenomics",
            "study": self.root_ot / "study",
        }

        self._create_views()

    def _glob(self, path: Path) -> str:
        return str(path / "**" / "*.parquet")

    def _create_views(self):
        for name, path in self.tables.items():
            if path.exists():
                self.con.execute(
                    f"""
                    CREATE OR REPLACE VIEW {name} AS
                    SELECT * FROM read_parquet('{self._glob(path)}')
                    """
                )
            else:
                print(f"Warning: missing table directory: {path}")

    def schema(self, table: str) -> pd.DataFrame:
        return self.con.execute(f"DESCRIBE {table}").df()
    
    # Resolve gene symbol or Ensembl ID
    def resolve_target(self, gene_or_ensembl: str) -> pd.DataFrame:
        q = gene_or_ensembl.strip()

        sql = """
        SELECT
            id AS target_id,
            approvedSymbol AS symbol,
            approvedName AS name,
            biotype
        FROM target
        WHERE
            id = ?
            OR upper(approvedSymbol) = upper(?)
        """

        return self.con.execute(sql, [q, q]).df()
    

    def show_schema_summary(self):
        for table in [
            "target",
            "disease",
            "drug_moa",
            "cgc",
            "reactome",
            "interactions",
            "pharmacogenomics",
            "study",
        ]:
            print("\n###", table)
            display(self.schema(table))

In [ ]:
ot = OpenTargetsLocal(root_ot=root_ot)

ot.resolve_target("TP53")

### Find a gene

In [ ]:
ot = OpenTargetsLocal(root_ot)

print(ot.con.execute("SHOW TABLES").df())

for table in ["target", "disease", "drug_moa", "cgc", "reactome", "interactions", "pharmacogenomics", "study"]:
    print("\n###", table)
    print(ot.con.execute(f"SELECT COUNT(*) AS n FROM {table}").df())

In [ ]:
ot.show_schema_summary()

### Then test target-specific retrieval with TP53:

In [ ]:
symbol = 'TP53'
symbol = 'PCYT2'
symbol = 'CCNB2'
symbol = 'BLM'

# Em adenocarcinoma pancreático, o gene KRAS mutado ativa uma proteína chamada KLF5, que coordena simultaneamente a desregulação de quatro enzimas da via de síntese de membranas celulares — 
# criando um perfil metabólico identificável por biópsia com implicações prognósticas e terapêuticas.
symbol = 'KLF5'
symbol = 'LPCAT1'

target = ot.resolve_target(symbol)
target_id = target.iloc[0]["target_id"]

target_id

### Test Cancer Gene Census

In [ ]:
df_cgc = ot.con.execute(
    """
    SELECT
        c.*,
        d.name AS disease_name, d.synonyms, d.ontology
    FROM cgc c
    LEFT JOIN disease d
        ON c.diseaseId = d.id
    WHERE c.targetId = ?
    ORDER BY c.score DESC NULLS LAST
    LIMIT 50
    """,
    [target_id]
).df()

print(len(df_cgc), 'diseases found.')
df_cgc.head(2).T

### Test Reactome evidence

In [ ]:
df_reactome = ot.con.execute(
    """
    SELECT
        r.*,
        d.name AS disease_name
    FROM reactome r
    LEFT JOIN disease d
        ON r.diseaseId = d.id
    WHERE r.targetId = ?
    ORDER BY r.score DESC NULLS LAST
    LIMIT 50
    """,
    [target_id]
).df()

print(len(df_reactome))
df_reactome.head(3).T

### drug mechanism of action

In [ ]:
display(ot.schema('drug_moa'))

In [ ]:
df_drug = ot.con.execute(
    """
    SELECT *
    FROM drug_moa
    WHERE CAST(targets AS VARCHAR) ILIKE ?
    """,
    [f"%{target_id}%"]
).df()

print(len(df_drug))
df_drug.head(3).T

### interactions

In [ ]:
ot.c("interactions")

In [ ]:
df_int = ot.con.execute(
    """
    SELECT *
    FROM interactions
    WHERE targetA = ?
       OR targetB = ?
    LIMIT 100
    """,
    [target_id, target_id]
).df()

print(len(df_int))
df_int.head(3).T